In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

*אומדן שימוש: דקה אחת על מעבד Eagle (הערה: זהו אומדן בלבד. זמן הריצה שלך עשוי להשתנות.)*

## רקע

Circuit-knitting הוא מונח גג המקיף שיטות שונות לחלוקת מעגל למספר מעגלי משנה קטנים יותר הכוללים פחות שערים ו/או קיוביטים. כל אחד ממעגלי המשנה יכול להתבצע באופן עצמאי והתוצאה הסופית מתקבלת על ידי עיבוד קלאסי מסוים על תוצאת כל מעגל משנה. טכניקה זו נגישה ב-[circuit cutting Qiskit addon](https://qiskit.github.io/qiskit-addon-cutting/index.html), הסבר מפורט של הטכניקה ניתן ב-[docs](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html) יחד עם [חומר מבוא](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html) נוסף.

מחברת זו עוסקת בשיטה הנקראת <b>חיתוך חוטים</b> שבה המעגל מחולק לאורך החוט [\[1\], \[2\]](#references). שימו לב שחלוקה פשוטה במעגלים קלאסיים מכיוון שהתוצאה בנקודת החלוקה ניתנת לקביעה דטרמיניסטית, והיא 0 או 1. עם זאת, מצב הקיוביט בנקודת החיתוך הוא, באופן כללי, מצב מעורב. לכן, יש למדוד כל מעגל משנה מספר פעמים בבסיסים שונים (בדרך כלל קבוצה שלמה טומוגרפית של בסיסים כגון בסיס Pauli [\[3\], \[4\]](#references) ובהתאם להכין אותו במצב העצמי שלו. האיור למטה (<i>באדיבות: עבודת דוקטורט, Ritajit Majumdar</i>) מציג דוגמה של חיתוך חוטים למצב GHZ של 4 קיוביטים לשלושה מעגלי משנה. כאן $M_j$ מציין קבוצת בסיסים (בדרך כלל Pauli X, Y ו-Z) ו-$P_i$ מציין קבוצת מצבים עצמיים (בדרך כלל $|0\rangle$, $|1\rangle$, $|+\rangle$ ו-$|+i\rangle$).

![wc-1.png](../docs/images/tutorials/wire-cutting-to-improve-performance/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting-to-improve-performance/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

מכיוון שלכל מעגל משנה יש פחות קיוביטים ו/או שערים, הם צפויים להיות פחות רגישים לרעש. מחברת זו מציגה דוגמה שבה ניתן להשתמש בשיטה זו לדיכוי יעיל של הרעש במערכת.

## דרישות

לפני שמתחילים את המדריך הזה, ודאו שהדברים הבאים מותקנים:

- Qiskit SDK v2.0 ומעלה, עם תמיכה ב-[visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 ומעלה ( `pip install qiskit-ibm-runtime` )
- Circuit cutting Qiskit addon v0.9.0 ומעלה (`pip install qiskit-addon-cutting`)

נשקול מעגל Many Body Localization (MBL) עבור מחברת זו. מעגל MBL הוא מעגל יעיל לחומרה והוא מפורמט על ידי שני פרמטרים $\theta$ ו-$\vec{\phi}$. כאשר $\theta$ מוגדר ל-$0$ והמצב ההתחלתי מוכן ב-$|0\rangle$ עבור כל הקיוביטים, ערך התוחלת האידיאלי של $\langle Z_i \rangle$ הוא $+1$ עבור כל אתר קיוביט $i$ ללא תלות בערכים של $\vec{\phi}$. ניתן לבדוק פרטים נוספים על מעגלי MBL ב-<a href="https://arxiv.org/abs/2307.07552">מאמר זה</a>.

## הגדרות

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## חלק I. דוגמה בקנה מידה קטן

### שלב 1: מיפוי קלטים קלאסיים לבעיה קוונטית

בתחילה אנחנו בונים מעגל תבנית ללא ערכי פרמטר ספציפיים. אנחנו גם מספקים מקומות שומרי מקום, הנקראים `CutWire`, כדי לסמן את מיקום החיתוכים. עבור הדוגמה בקנה מידה קטן אנחנו שוקלים מעגל MBL של 10 קיוביטים.

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

We calculate the average expectation value $O = \frac{1}{n} \sum_i Z_i$ over all qubits for $\theta = 0$. Since the ideal expectation value of $\langle Z_i \rangle = 1$ $\forall$ $i$, the ideal expectation value of $O$ is also $1$. The parameters $\phi$ are selected randomly.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

כעת אנחנו מסמנים את המעגל לחיתוך על ידי הוספת **CutWire** מתאים ליצירת שני חיתוכים שווים בערך. אנחנו מגדירים `use_cut=True` בפונקציה, ומאפשרים לה לסמן אחרי $\frac{n}{2}$ קיוביטים, כאשר $n$ הוא מספר הקיוביטים במעגל המקורי.

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/31844134-514b-46ea-85f9-133e432f053f-0.avif)

### שלב 2: אופטימיזציה של הבעיה לביצוע על חומרה קוונטית
לאחר מכן אנחנו חותכים את המעגל לשני מעגלי משנה קטנים יותר. עבור דוגמה זו, אנחנו נצמדים ל-2 מעגלי משנה בלבד. לשם כך, אנחנו משתמשים ב-<a href="https://qiskit.github.io/qiskit-addon-cutting/">Qiskit Addon: Circuit Cutting</a>.
#### חיתוך המעגל למעגלי משנה קטנים יותר
חיתוך החוט בנקודה מגדיל את מספר הקיוביטים באחד. מלבד הקיוביט המקורי, יש כעת קיוביט נוסף כמקום שומר מקום למעגל לאחר החיתוך. התמונה הבאה נותנת ייצוג:

![wc-4.png](../docs/images/tutorials/wire-cutting-to-improve-performance/dfc5f923-e507-4873-888e-d90e1618be3a.avif)

Addon זה משתמש בפונקציה `cut_wires` כדי לתת מענה לקיוביטים הנוספים הנוצרים כתוצאה מהחיתוך.

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

#### יצירה והרחבת הנצפים
כעת אנחנו בונים את הנצפה $M_z = \frac{1}{n}\sum_{i=1}^n \langle Z_i \rangle$. מכיוון שהתוצאה האידיאלית של $\langle Z_i \rangle$ עבור כל $i$ היא $+1$, התוצאה האידיאלית של $M_z$ היא גם $+1$.

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

עם זאת, שימו לב שמספר הקיוביטים במעגל גדל לאחר הוספת פעולות `Move` וירטואליות של 2 קיוביטים לאחר החיתוך. לכן, עלינו להרחיב גם את הנצפים על ידי הוספת זהויות כדי להתאים למעגל הנוכחי.

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

הבה נצפה במעגלי המשנה

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

#### Transpile the circuits onto the backend

For the first example involving only simulation, we transpile the circuit into the basis gate set of the backend:

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/35920640-76e8-4af6-a252-ee6a22e9c26a-0.avif)

הנצפים חולקו גם כן כדי להתאים למעגלי המשנה

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

שימו לב שכל מעגל משנה מוביל למספר דגימות. השחזור לוקח בחשבון את התוצאה של כל אחת מהדגימות הללו. כל אחת מהדגימות הללו נקראת `subexperiment`.
הרחבת הנצפה באמצעות פעולת `Move` דורשת מבנה נתונים `PauliList`. אנחנו יכולים גם ליצור את הנצפה $M_z$ במבנה הנתונים הכללי יותר `SparsePauliOp` שיהיה שימושי מאוחר יותר במהלך שחזור ה-subexperiments.

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

הבה נראה שתי דוגמאות שבהן הקיוביטים החתוכים נמדדים בשני בסיסים שונים. ראשית, הוא נמדד בבסיס Z רגיל, ולאחר מכן הוא נמדד בבסיס X.

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/987547e4-296a-41e4-ad82-41f4139a87a0-0.avif)

#### העברת כל subexperiment להתאמה לחומרה

כרגע עלינו להעביר את המעגלים שלנו להתאמה לחומרה לפני הגשתם לביצוע. לכן, נעביר כל מעגל ב-subexperiments תחילה להתאמה לחומרה.